In [ ]:
#!/usr/bin/env python3

def extract_paper_titles(input_file, output_file):
    """
    Extract paper titles from citations.txt file.
    Handles format where lines alternate between:
    - Numbers (like '329', '330')
    - Paper titles 
    - Author names
    """
    titles = []
    
    try:
        with open(input_file, 'r', encoding='utf-8') as file:
            lines = [line.strip() for line in file.readlines() if line.strip()]
        
        i = 0
        while i < len(lines):
            line = lines[i]
            
            # Skip lines that are just numbers (citation numbers)
            if line.isdigit():
                i += 1
                continue
            
            # Skip lines that look like arXiv identifiers
            if line.startswith(('20', 'arXiv', 'http')) and any(char.isdigit() for char in line):
                i += 1
                continue
            
            # Skip author lines (contain names with commas and "and X more")
            if (',' in line and (' and ' in line.lower() or 'more' in line.lower())) or \
               (line.count(',') >= 2):  # Multiple commas usually indicate author list
                i += 1
                continue
            
            # Skip pure numbers or short numeric strings
            if line.replace(',', '').replace("'", "").strip().isdigit():
                i += 1
                continue
            
            # If we get here, it's likely a paper title
            # Clean up any quotes or commas from the beginning/end
            clean_title = line.strip("',\" ")
            
            # Additional check: titles are usually longer than 10 characters
            if len(clean_title) > 10 and not clean_title.isdigit():
                titles.append(clean_title)
            
            i += 1
        
        # Write titles to output file
        with open(output_file, 'w', encoding='utf-8') as file:
            for title in titles:
                file.write(title + '\n')
        
        print(f"Extracted {len(titles)} paper titles")
        print(f"Saved to: {output_file}")
        
        # Print first few titles as preview
        print("\nFirst 10 titles:")
        for i, title in enumerate(titles[:10], 1):
            print(f"{i}. {title}")
            
    except FileNotFoundError:
        print(f"Error: File '{input_file}' not found")
    except Exception as e:
        print(f"Error: {e}")

def extract_titles_simple_pattern(input_file, output_file):
    """
    Alternative approach: Extract every 3rd line starting from index 1
    (assuming pattern: number, title, authors, number, title, authors...)
    """
    titles = []
    
    try:
        with open(input_file, 'r', encoding='utf-8') as file:
            lines = [line.strip().strip("',\" ") for line in file.readlines() if line.strip()]
        
        # Extract every 3rd line starting from index 1 (0-indexed)
        # Pattern: [0]=number, [1]=title, [2]=authors, [3]=number, [4]=title, [5]=authors...
        for i in range(1, len(lines), 3):
            if i < len(lines):
                title = lines[i].strip("',\" ")
                if title and len(title) > 10:  # Basic validation
                    titles.append(title)
        
        # Write titles to output file
        with open(output_file, 'w', encoding='utf-8') as file:
            for title in titles:
                file.write(title + '\n')
        
        print(f"Extracted {len(titles)} paper titles using pattern method")
        print(f"Saved to: {output_file}")
        
        # Print first few titles as preview
        print("\nFirst 10 titles:")
        for i, title in enumerate(titles[:10], 1):
            print(f"{i}. {title}")
            
    except FileNotFoundError:
        print(f"Error: File '{input_file}' not found")
    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    input_filename = "citations.txt"
    output_filename = "paper_titles.txt"
    
    print("Method 1: Smart filtering")
    extract_paper_titles(input_filename, output_filename)
    
    print("\n" + "="*50 + "\n")
    
    print("Method 2: Pattern-based extraction")
    extract_titles_simple_pattern(input_filename, "paper_titles_pattern.txt")
    
    print("\nTry both methods and see which one works better for your data!")

In [1]:
paper_titles  = open('paper_titles.txt','r').read().split('\n')

In [14]:
#!/usr/bin/env python3

import pandas as pd
import openai
import time
import os
from typing import List, Dict
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import json

def classify_paper_title(title: str, client) -> Dict:
    """
    Use OpenAI API to classify if a paper title is related to:
    1. Reducing the length of chain of thought
    2. Improving reasoning
    """
    
    prompt = f"""
    Analyze the following research paper title and determine if it's related to either:
    1. Reducing the length of chain of thought (making reasoning more concise/efficient)
    2. Improving reasoning capabilities in AI/ML models
    
    Paper title: "{title}"
    
    Please respond with one word containing the right category:
    "chain_of_thought_reduction", "reasoning_improvement", "both", or "neither"
    
    Focus on papers about:
    - Making chain of thought more efficient/shorter
    - Improving logical reasoning, mathematical reasoning, or problem-solving
    - Optimization of reasoning processes
    - Better prompting techniques for reasoning
    - Reasoning evaluation and enhancement
    """
    
    max_retries = 3
    retry_delay = 1.0
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are an expert in AI/ML research papers, particularly in reasoning and chain of thought techniques."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.001,
                max_tokens=10
            )
            
            # Try to parse the JSON response
            return {
                'title': title,
                'category': str(response.choices[0].message.content),
            }
            
        except json.JSONDecodeError:
            # If JSON parsing fails, create a manual classification
            content = response.choices[0].message.content.lower()
            is_relevant = any(keyword in content for keyword in ['true', 'relevant', 'yes', 'reasoning', 'chain'])
            
            return {
                'title': title,
                'category': "uncertain",
            }
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay * (2 ** attempt))  # Exponential backoff
                continue
            else:
                return {
                    'title': title,
                    'category': "error",
                }

In [15]:
client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
classify_paper_title('SEAL: Steerable Reasoning Calibration of Large Language Models for Free', client)

{'title': 'SEAL: Steerable Reasoning Calibration of Large Language Models for Free',
 'category': 'reasoning_improvement'}

In [17]:
def classify_batch(titles_batch: List[str], client, pbar) -> List[Dict]:
    """Process a batch of titles"""
    results = []
    for title in titles_batch:
        result = classify_paper_title(title, client)
        results.append(result)
        pbar.update(1)
    return results

def main():
    # Initialize OpenAI client
    client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    # Read paper titles
    try:
        paper_titles = open('paper_titles.txt', 'r').read().split('\n')
        paper_titles = [title.strip() for title in paper_titles if title.strip()]
        print(f"Found {len(paper_titles)} paper titles")
    except FileNotFoundError:
        print("Error: paper_titles.txt not found")
        return
    
    # Configuration for parallel processing
    MAX_WORKERS = 10  # Adjust based on your OpenAI rate limits
    BATCH_SIZE = 5   # Number of titles per worker batch
    
    print(f"Starting parallel classification with {MAX_WORKERS} workers...")
    
    # Create batches of titles
    title_batches = []
    for i in range(0, len(paper_titles), BATCH_SIZE):
        batch = paper_titles[i:i + BATCH_SIZE]
        title_batches.append(batch)
    
    # Process batches in parallel
    all_results = []
    
    with tqdm(total=len(paper_titles), desc="Classifying papers") as pbar:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submit all batches
            future_to_batch = {
                executor.submit(classify_batch, batch, client, pbar): batch 
                for batch in title_batches
            }
            
            # Collect results as they complete
            for future in as_completed(future_to_batch):
                try:
                    batch_results = future.result()
                    all_results.extend(batch_results)
                except Exception as e:
                    batch = future_to_batch[future]
                    print(f"Error processing batch: {e}")
                    # Add error results for failed batch
                    for title in batch:
                        all_results.append({
                            'title': title,
                            'category': "error",
                        })
                        pbar.update(1)
    
    # Sort results by original order (optional)
    title_to_index = {title: i for i, title in enumerate(paper_titles)}
    # Create DataFrames
    papers_df = pd.DataFrame(all_results)
    
    # Save results
    papers_df.to_csv('papers_df.csv', index=False)
    
    return papers_df

if __name__ == "__main__":
    # Make sure to set your OpenAI API key first!
    if not os.getenv('OPENAI_API_KEY'):
        print("Please set your OpenAI API key:")
        print("  export OPENAI_API_KEY='your-key-here'")
        print("Or uncomment and set the api_key directly in the script")
        exit(1)
    
    start_time = time.time()
    papers_df = main()
    end_time = time.time()
    
    print(f"\nTotal processing time: {end_time - start_time:.2f} seconds")

Found 2456 paper titles
Starting parallel classification with 10 workers...


Classifying papers: 100%|███████████████████| 2456/2456 [03:10<00:00, 12.86it/s]


Total processing time: 191.02 seconds


In [23]:
papers_df['category'] = papers_df['category'].str.replace('"', '').str.replace("'", '')

In [31]:
papers_df.to_csv('papers_df.csv')

In [30]:
papers_df_subset = papers_df[(papers_df['category'] == 'both') | (papers_df['category'] == 'chain_of_thought_reduction')]

In [33]:
papers_df_subset.to_csv('papers_df_subset.csv')

In [34]:
papers_df_subset

,title,category
21,Accelerating Chain-of-Thought Reasoning: When ...,both
40,Hunyuan-TurboS: Advancing Large Language Model...,both
52,Communication-Efficient Hybrid Language Model ...,chain_of_thought_reduction
58,A Survey on Inference Engines for Large Langua...,both
63,Reasoning with OmniThought: A Large CoT Datase...,both
...,...,...
2340,RaCT: Ranking-aware Chain-of-Thought Optimizat...,both
2408,SelectLLM: Query-Aware Efficient Selection Alg...,chain_of_thought_reduction
2435,RePrompt: Planning by Automatic Prompt Enginee...,both
2438,Towards Better Chain-of-Thought: A Reflection ...,both


In [35]:
for idx,row in papers_df.iterrows():
    if 'SEAL' in ro:
        print(title)

SEAL: Steerable Reasoning Calibration of Large Language Models for Free
